# Viterbi algorithm
The Viterbi algorithm is a dynamic programming algorithm that finds the most likely sequence of hidden events that would explain a sequence of observed events. The result of the algorithm is often called the Viterbi path. It is most commonly used with hidden Markov models (HMMs). For example, if a doctor observes a patient's symptoms over several days (the observed events), the Viterbi algorithm could determine the most probable sequence of underlying health conditions (the hidden events) that caused those symptoms. 

In [ ]:
import dataclasses
import pandas as pd
import ipywidgets # type: ignore
import random
import time
import math
from IPython import display

!python --version

Python 3.10.12


In [ ]:


def row_normalize(df):
  return df.div(df.sum(axis=1), axis=0)

sentences = [
    'we/PRON saw/VERB her/PRON duck/NOUN',
    'we/PRON duck/VERB a/DET saw/NOUN',
    'a/DET duck/NOUN saw/VERB a/DET duck/NOUN',
]
smoothing = 0.1
sentences = [[tuple(w.split('/')) for w in sentence.split(' ')] for sentence in sentences]

tags = list(set([word[1] for sentence in sentences for word in sentence]))
words = list(set([word[0] for sentence in sentences for word in sentence]))

emissions = pd.DataFrame([[smoothing] * len(words)] * len(tags), index=tags, columns=words)
for sentence in sentences:
  for word_tag in sentence:
    word, tag = word_tag
    emissions[word][tag] += 1
emissions = row_normalize(emissions)

transitions = pd.DataFrame([[smoothing] * len(tag)] * (len(tags) + 1), index=[None] + tags, columns=tags)
for sentence in sentences:
  for w1, w2 in zip([(None, None)] + sentence, sentence):
    t1 = w1[1]
    t2 = w2[1]
    transitions[t2][t1] += 1
transitions = row_normalize(transitions)

transitions

# emissions[word][tag] = p(word | tag)
# transitions[tag1][tag2] = p(tag1 | tag2)
# transitions['DET'][None] = p(DET) as the initial word.


,NOUN,DET,VERB,PRON
None,0.029412,0.323529,0.029412,0.617647
NOUN,0.071429,0.071429,0.785714,0.071429
DET,0.911765,0.029412,0.029412,0.029412
VERB,0.029412,0.617647,0.029412,0.323529
PRON,0.323529,0.029412,0.617647,0.029412


In [ ]:
emissions

,duck,a,her,saw,we
NOUN,0.688889,0.022222,0.022222,0.244444,0.022222
DET,0.028571,0.885714,0.028571,0.028571,0.028571
VERB,0.314286,0.028571,0.028571,0.600000,0.028571
PRON,0.028571,0.028571,0.314286,0.028571,0.600000


In [ ]:
def highlight_cells(col, highlights):
  results = [''] * len(col)
  col_index = list(col.index)
  for highlight in highlights:
    if col.name == highlight[0]:
      results[col_index.index(highlight[1])] = "background-color: yellow"
  return results


@dataclasses.dataclass
class Viterbi():
  sentence: list
  tagging: list
  back_pointers: pd.DataFrame
  best_score: pd.DataFrame
  numbered_words: list
  emissions: pd.DataFrame
  transitions: pd.DataFrame

  # For visualization
  highlights: list
  back_highlights: list

  def __init__(self, emissions, trnasitions, sentence):
    super().__init__()
    self.emissions = emissions
    self.transitions = trnasitions
    self.sentence = sentence
    self.tagging = [None] * len(sentence)
    for word in sentence:
      if word not in emissions.columns:
        raise ValueError(f'Word {word} not in known words ({list(emissions.columns)}).')
    num_words = len(sentence)
    tags = list(emissions.index)
    num_tags = len(emissions.index)
    self.numbered_words = [f'{w}_{i}' for i, w in enumerate(sentence)]
    self.best_score = pd.DataFrame([[float('-inf')] * num_words] * num_tags,
                                   columns=self.numbered_words, index=tags)
    self.back_pointers = pd.DataFrame([[''] * num_words] * num_tags,
                                   columns=self.numbered_words, index=tags)
    self.highlights = []
    self.back_highlights = []

  def _ipython_display_(self):
    display.display(self.sentence)
    display.display(self.tagging)
    if self.best_score is not None:
      display.display(self.best_score.style.apply(highlight_cells, highlights=self.highlights))
    display.display(self.back_pointers.style.apply(highlight_cells, highlights=self.back_highlights))

  def step(self):
    for position in range(len(self.sentence)):
      word = self.sentence[position]
      numbered_word = self.numbered_words[position]
      if position > 0:
        prev_numbered_word = self.numbered_words[position - 1]
      for curr_tag in self.transitions.columns:
        for prev_tag in self.transitions.index:
          transition = self.transitions[curr_tag][prev_tag]
          emission = self.emissions[word][curr_tag]
          if position == 0 and prev_tag is None:
            curr_score = math.log(transition) + math.log(emission)
            self.best_score[numbered_word][curr_tag] = curr_score
            self.highlights = [(numbered_word, curr_tag)]
            yield
          if position > 0 and prev_tag is not None:
            curr_score = (math.log(transition) + math.log(emission) +
                          self.best_score[prev_numbered_word][prev_tag])
            if curr_score > self.best_score[numbered_word][curr_tag]:
              self.best_score[numbered_word][curr_tag] = curr_score
              self.back_pointers[numbered_word][curr_tag] = prev_tag
              self.highlights = [(numbered_word, curr_tag),
                                 (prev_numbered_word, prev_tag)]
              yield
    self.highlights = []
    best_last = self.best_score[numbered_word].idxmax()
    for position in range(len(self.sentence) - 1, -1, -1):
      numbered_word = self.numbered_words[position]
      self.tagging[position] = best_last
      self.back_highlights = [(numbered_word, best_last)]
      best_last = self.back_pointers[numbered_word][best_last]
      yield
    # Write me
    yield


v = Viterbi(emissions, transitions, 'we saw her duck'.split(' '))

button = ipywidgets.Button(description='Step')

iter = v.step()
for i in range(30):
  next(iter)
def button_clicked(b):
  try:
    next(iter)
  except StopIteration:
    pass
  except Exception as e:
    raise e
  display.clear_output(wait=True)
  display.display(button)
  display.display(v)

button.on_click(button_clicked)

display.display(button)
display.display(v)


Button(description='Step', style=ButtonStyle())

['we', 'saw', 'her', 'duck']

['PRON', 'VERB', 'PRON', 'NOUN']

,we_0,saw_1,her_2,duck_3
NOUN,-7.333023,-3.529896,-9.318350,-5.772386
DET,-4.683813,-8.074372,-6.022514,-11.352954
VERB,-7.081709,-1.985327,-7.326406,-5.910536
PRON,-0.992664,-8.074372,-4.271245,-11.352954


,we_0,saw_1,her_2,duck_3
NOUN,,PRON,VERB,PRON
DET,,PRON,VERB,PRON
VERB,,PRON,NOUN,PRON
PRON,,PRON,VERB,PRON
